# Pharma Domain Causal LM Fine-Tuning with QLoRA

<span style="color:green"><b>Concept:</b> This notebook performs <b>non-instruction fine-tuning</b> (also called continued pretraining / domain adaptation) on raw pharma text extracted from a PDF.</span>

<span style="color:green"><b>Important:</b> The model is trained to <b>predict the next token</b> from domain text. It is <b>not</b> trained in question-answer or chatbot format.</span>

## Objective

We will:

1. Extract raw text from a pharma PDF
2. Clean and normalize the text
3. Convert it into a Hugging Face dataset
4. Tokenize and pack text into fixed-length blocks
5. Load a pretrained base model
6. Add a LoRA adapter
7. Train only the adapter using QLoRA-friendly settings
8. Save the adapter
9. Reload the base model + adapter for inference
10. Generate pharma-style text continuations

## Pipeline

```text
Pharma PDF
   ↓
PDF text extraction
   ↓
Text cleaning and normalization
   ↓
Data creation
   ↓
Hugging Face Dataset Conversion
   ↓
Tokenization
   ↓
LoRA/QLoRA fine-tuning
   ↓
Validation loss
   ↓
Adapter saving and reloading
   ↓
Text continuation inference
```

In [ ]:
# ============================================================
# 1. Install dependencies
# ============================================================
# PyMuPDF      -> extract text from PDF
# datasets     -> Hugging Face dataset utilities
# transformers -> tokenizer, model, trainer
# accelerate   -> training backend support
# peft         -> LoRA / PEFT support
# bitsandbytes -> 4-bit quantized loading for QLoRA
# sentencepiece-> tokenizer dependency for many LLMs

!pip install -q -U pymupdf datasets transformers accelerate peft bitsandbytes sentencepiece

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 25.0/25.0 MB 53.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 555.1/555.1 kB 22.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11.2/11.2 MB 46.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 389.2/389.2 kB 12.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.7/60.7 MB 9.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 48.9/48.9 MB 9.2 MB/s eta 0:00:00


## 2. Imports

<span style="color:green"> Import only the libraries needed for this fine-tuning pipeline.</span>

In [ ]:
# ============================================================
# 2. Imports
# ============================================================

import os
import re
import gc
import math
import json
import random
import unicodedata
from dataclasses import dataclass, asdict
from typing import List, Dict, Any

import fitz  # PyMuPDF
import torch
from datasets import Dataset, DatasetDict

from transformers import (
    AutoTokenizer,
    AutoModelForCausalLM,
    BitsAndBytesConfig,
    DataCollatorForLanguageModeling,
    Trainer,
    TrainingArguments,
    set_seed,
)

from peft import (
    LoraConfig,
    TaskType,
    get_peft_model,
    prepare_model_for_kbit_training,
    PeftModel,
)

In [34]:
from google.colab import userdata
READ_TOKEN=userdata.get("HF_TOKEN_READ")
WRITE_TOKEN=userdata.get('HF_TOKEN_WRITE')
# import os
# os.environ["HF_TOKEN_READ"] = userdata.get("HF_TOKEN_READ")
# os.environ["HF_TOKEN_WRITE"] = userdata.get("HF_TOKEN_WRITE")

## 3. Configuration

<span style="color:green"><b>Concept:</b> Keep all important settings in one place so the notebook is easy to maintain, debug, and reproduce.</span>

In [ ]:
# ============================================================
# 3. Configuration
# ============================================================

CONFIG = {
    "pdf_path": "/content/Metformin-Lipid-Therapy-Knowledge.pdf",  # Path of the input PDF file
    "model_name": "TinyLlama/TinyLlama-1.1B-intermediate-step-1431k-3T",  # Pretrained base model name

    "output_dir": "/content/pharma_tinyllama_lora_output",  # Folder for training outputs/checkpoints
    "adapter_dir": "/content/pharma_tinyllama_lora_adapter",  # Folder to save final LoRA adapter
    "processed_data_dir": "/content/pharma_processed_data",  # Folder to save cleaned/processed text files

    "min_chars_per_paragraph": 80,  # Ignore paragraphs shorter than this many characters
    "block_size": 512,  # Number of tokens per training sequence/block

    "test_size": 0.15,  # Fraction of dataset used for validation
    "seed": 42,  # Random seed for reproducibility

    "lora_r": 16,  # LoRA rank; controls adapter capacity
    "lora_alpha": 32,  # LoRA scaling factor; controls adapter strength
    "lora_dropout": 0.05,  # Dropout inside LoRA layers to reduce overfitting

    "num_train_epochs": 3.0,  # Number of full passes through training data
    "per_device_train_batch_size": 1,  # Training batch size per device
    "per_device_eval_batch_size": 1,  # Evaluation batch size per device
    "gradient_accumulation_steps": 8,  # Accumulate gradients to simulate larger batch size
    "learning_rate": 2e-4,  # Optimizer learning rate
    "weight_decay": 0.01,  # Regularization strength to reduce overfitting
    "logging_steps": 5,  # Log training progress every N steps
    "eval_steps": 10,  # Run evaluation every N steps
    "save_steps": 25,  # Save checkpoint every N steps
    "save_total_limit": 2,  # Keep only this many recent checkpoints
    "max_steps": -1,  # -1 means train using epochs, not fixed step count
}

In [ ]:
# ============================================================
# 3. Global configuration
# ============================================================
# Keep all important parameters in one place.
# This makes the notebook easier to debug, reproduce, and productionize.

from dataclasses import dataclass, asdict

@dataclass
class Config:
    # Path of the pharma PDF file that will be used as the raw domain corpus.
    pdf_path: str = "/content/Metformin-Lipid-Therapy-Knowledge.pdf"

    # Base causal language model that we will fine-tune on pharma-domain text.
    model_name: str = "TinyLlama/TinyLlama-1.1B-intermediate-step-1431k-3T"

    # Directory where training checkpoints will be saved during fine-tuning.
    output_dir: str = "/content/pharma_tinyllama_lora_output"

    # Directory where the final trained LoRA adapter will be saved.
    adapter_dir: str = "/content/pharma_tinyllama_lora_adapter"

    # Directory where cleaned and processed training data will be saved.
    processed_data_dir: str = "/content/pharma_processed_data"

    # Minimum paragraph length required to keep a paragraph for training.
    min_chars_per_paragraph: int = 80

    # Number of tokens in each training block for causal language modeling.
    block_size: int = 512

    # Percentage of data used for validation instead of training.
    test_size: float = 0.15

    # Random seed used to make splitting and training more reproducible.
    seed: int = 42

    # LoRA rank; controls the size and capacity of the trainable adapter.
    lora_r: int = 16

    # LoRA scaling factor; controls the strength of the LoRA update.
    lora_alpha: int = 32

    # Dropout applied inside LoRA layers to reduce overfitting.
    lora_dropout: float = 0.05

    # Number of times the model will see the complete training dataset.
    num_train_epochs: float = 3.0

    # Number of training samples processed per GPU/device at one time.
    per_device_train_batch_size: int = 1

    # Number of validation samples processed per GPU/device at one time.
    per_device_eval_batch_size: int = 1

    # Number of small batches accumulated before one optimizer update.
    gradient_accumulation_steps: int = 8

    # Step size used by the optimizer to update trainable LoRA weights.
    learning_rate: float = 2e-4

    # Fraction of early training steps used to gradually increase learning rate.
    warmup_ratio: float = 0.03

    # Regularization value used to prevent weights from becoming too large.
    weight_decay: float = 0.01

    # Number of training steps after which logs will be printed.
    logging_steps=1
    logging_first_step=True

    # Number of training steps after which validation will be performed.
    eval_steps: int = 10

    # Number of training steps after which a checkpoint will be saved.
    save_steps: int = 25

    # Maximum number of checkpoints to keep; older checkpoints will be deleted.
    save_total_limit: int = 2

    # Maximum number of training steps; -1 means train using num_train_epochs.
    max_steps: int = -1

In [ ]:
# Create directories
import os
os.makedirs(CONFIG["output_dir"], exist_ok=True)
os.makedirs(CONFIG["adapter_dir"], exist_ok=True)
os.makedirs(CONFIG["processed_data_dir"], exist_ok=True)

print(json.dumps(CONFIG, indent=2))

{
  "pdf_path": "/content/Metformin-Lipid-Therapy-Knowledge.pdf",
  "model_name": "TinyLlama/TinyLlama-1.1B-intermediate-step-1431k-3T",
  "output_dir": "/content/pharma_tinyllama_lora_output",
  "adapter_dir": "/content/pharma_tinyllama_lora_adapter",
  "processed_data_dir": "/content/pharma_processed_data",
  "min_chars_per_paragraph": 80,
  "block_size": 512,
  "test_size": 0.15,
  "seed": 42,
  "lora_r": 16,
  "lora_alpha": 32,
  "lora_dropout": 0.05,
  "num_train_epochs": 3.0,
  "per_device_train_batch_size": 1,
  "per_device_eval_batch_size": 1,
  "gradient_accumulation_steps": 8,
  "learning_rate": 0.0002,
  "weight_decay": 0.01,
  "logging_steps": 5,
  "eval_steps": 10,
  "save_steps": 25,
  "save_total_limit": 2,
  "max_steps": -1
}


## 4. Check PDF

<span style="color:green">Before training, confirm the PDF exists at the expected location.</span>

In [7]:
# ============================================================
# 4. Check PDF
# ============================================================

if not os.path.exists(CONFIG["pdf_path"]):
    raise FileNotFoundError(f"PDF not found at: {CONFIG['pdf_path']}")
else:
    print(f"PDF found: {CONFIG['pdf_path']}")

PDF found: /content/Metformin-Lipid-Therapy-Knowledge.pdf


## 5. Extract text from PDF

<span style="color:green">Extract text page by page so we can inspect and audit the PDF extraction quality.</span>

In [8]:
# ============================================================
# 5. Extract text from PDF
# ============================================================

def extract_pdf_pages(pdf_path):
    pages = []
    with fitz.open(pdf_path) as doc:
        for page_index, page in enumerate(doc, start=1):
            text = page.get_text("text")
            text = text.strip() if text else ""
            if text:
                pages.append({
                    "page": page_index,
                    "text": text,
                    "char_count": len(text),
                })
    return pages

In [9]:
pdf_pages = extract_pdf_pages(CONFIG["pdf_path"])

print(f"Total pages with text: {len(pdf_pages)}")
for item in pdf_pages:
    print(f"Page {item['page']}: {item['char_count']} characters")
print("Raw page 1 preview:\n")
print(pdf_pages[0]["text"][:2000])

Total pages with text: 6
Page 1: 2244 characters
Page 2: 2889 characters
Page 3: 2636 characters
Page 4: 2416 characters
Page 5: 2613 characters
Page 6: 2761 characters
Raw page 1 preview:

Metformin is one of the most widely prescribed oral antihyperglycemic agents.​
 Its primary mechanism of action involves the activation of AMP-activated protein kinase 
(AMPK), a central metabolic regulator that promotes glucose uptake and fatty acid oxidation 
while inhibiting hepatic gluconeogenesis.​
 Beyond its glycemic control, Metformin has been shown to improve cardiovascular outcomes 
and display anti-inflammatory properties.​
 Recent studies also suggest potential anticancer effects through inhibition of the mTOR 
signaling pathway and suppression of tumor angiogenesis. 
 
Clinical trials have demonstrated that combining Atorvastatin with Ezetimibe results in 
significant reductions in low-density lipoprotein cholesterol (LDL-C) levels compared to 
monotherapy.​
 Ezetimibe acts by inhibitin

## 6. Clean PDF text

<span style="color:green">PDF text usually contains hidden characters, broken line wraps, and formatting noise. We clean it before tokenization.</span>

| Cleaning Step                          | Code / Logic                             | What It Does                                                                  | Example Before                                                  | Example After                                                  | Why It Matters for Fine-Tuning                                            |
| -------------------------------------- | ---------------------------------------- | ----------------------------------------------------------------------------- | --------------------------------------------------------------- | -------------------------------------------------------------- | ------------------------------------------------------------------------- |
| Unicode normalization                  | `unicodedata.normalize("NFKC", text)`    | Converts unusual Unicode characters into standard readable characters.        | `ＡＭＰＫ`, `ﬁ`                                                     | `AMPK`, `fi`                                                   | Prevents tokenizer confusion caused by hidden or non-standard characters. |
| Remove zero-width characters           | `text.replace("\u200b", "")`             | Removes invisible zero-width spaces from PDF text.                            | `Metformin​ activates AMPK`                                     | `Metformin activates AMPK`                                     | Invisible characters can create bad tokens and noisy training data.       |
| Remove BOM / hidden marker             | `text.replace("\ufeff", "")`             | Removes hidden Byte Order Mark characters sometimes found in extracted text.  | `﻿Metformin is used...`                                         | `Metformin is used...`                                         | Keeps the training text clean and consistent.                             |
| Fix hyphenated line breaks             | `re.sub(r"(\w)-\n(\w)", r"\1\2", text)`  | Joins words that were broken across PDF lines.                                | `gluconeogene-\nsis`                                            | `gluconeogenesis`                                              | Prevents the model from learning broken medical terms.                    |
| Normalize spaces and tabs              | `re.sub(r"[ \t]+", " ", text)`           | Converts multiple spaces or tabs into one space.                              | `Metformin     activates    AMPK`                               | `Metformin activates AMPK`                                     | Makes text consistent and easier for tokenizer/model to learn.            |
| Normalize blank lines                  | `re.sub(r"\n{3,}", "\n\n", text)`        | Converts too many blank lines into a proper paragraph gap.                    | `Para 1\n\n\n\nPara 2`                                          | `Para 1\n\nPara 2`                                             | Preserves paragraph structure without unnecessary whitespace noise.       |
| Remove standalone page numbers         | `re.sub(r"(?m)^\s*\d+\s*$", "", text)`   | Removes lines that contain only page numbers.                                 | `1` or `23`                                                     | Removed                                                        | Prevents the model from learning irrelevant PDF page numbers.             |
| Split into paragraphs                  | `re.split(r"\n\s*\n", text)`             | Splits text wherever there is a blank line.                                   | `Para 1\n\nPara 2`                                              | `["Para 1", "Para 2"]`                                         | Helps preserve meaningful document structure.                             |
| Remove line wrapping inside paragraphs | `re.sub(r"\n+", " ", paragraph)`         | Converts broken lines inside the same paragraph into a single paragraph line. | `Metformin is widely prescribed\noral antihyperglycemic agent.` | `Metformin is widely prescribed oral antihyperglycemic agent.` | Prevents the model from learning artificial PDF line breaks.              |
| Normalize paragraph spacing            | `re.sub(r"\s+", " ", paragraph).strip()` | Removes extra spaces inside each paragraph and trims start/end spaces.        | `  Metformin   activates   AMPK.  `                             | `Metformin activates AMPK.`                                    | Produces clean, readable training examples.                               |
| Remove empty paragraphs                | `if paragraph:`                          | Keeps only non-empty cleaned paragraphs.                                      | `""`                                                            | Removed                                                        | Avoids useless blank samples in the dataset.                              |
| Rebuild cleaned text                   | `"\n\n".join(cleaned_paragraphs)`        | Joins cleaned paragraphs with two newlines.                                   | List of cleaned paragraphs                                      | Clean paragraph-level text                                     | Creates a clean corpus suitable for causal LM training.                   |
| Track cleaned page length              | `char_count: len(cleaned_text)`          | Stores number of characters after cleaning.                                   | Raw page length unknown                                         | `char_count = 1450`                                            | Helps debug whether a page has too little or too much extracted content.  |
| Preview cleaned output                 | `cleaned_pages[0]["text"][:1500]`        | Prints first 1500 characters of cleaned page 1.                               | Full cleaned page                                               | Preview text                                                   | Helps manually verify that cleaning worked correctly.                     |


In [10]:
# ============================================================
# 6. Text cleaning
# ============================================================

def normalize_unicode(text):
    # Normalize strange Unicode forms into standard readable forms
    return unicodedata.normalize("NFKC", text)


def clean_pdf_text(text):
    # Normalize Unicode characters
    text = normalize_unicode(text)

    # Remove hidden characters often found in PDF extraction
    text = text.replace("\u200b", "")   # zero-width space
    text = text.replace("\ufeff", "")   # BOM / hidden marker

    # Fix words broken across lines: gluconeogene-\nsis -> gluconeogenesis
    text = re.sub(r"(\w)-\n(\w)", r"\1\2", text)

    # Normalize spaces and tabs
    text = re.sub(r"[ \t]+", " ", text)

    # Normalize too many blank lines
    text = re.sub(r"\n{3,}", "\n\n", text)

    # Remove standalone page-number-like lines
    text = re.sub(r"(?m)^\s*\d+\s*$", "", text)

    # Split by paragraph boundary
    paragraphs = re.split(r"\n\s*\n", text)

    cleaned_paragraphs = []
    for paragraph in paragraphs:
        # Remove line breaks inside the same paragraph
        paragraph = re.sub(r"\n+", " ", paragraph)
        paragraph = re.sub(r"\s+", " ", paragraph).strip()

        if paragraph:
            cleaned_paragraphs.append(paragraph)

    return "\n\n".join(cleaned_paragraphs)

In [11]:
cleaned_pages = []

for page in pdf_pages:
    cleaned_text = clean_pdf_text(page["text"])
    cleaned_pages.append({
        "page": page["page"],
        "text": cleaned_text,
        "char_count": len(cleaned_text),
    })

In [12]:
print("Cleaned page 1 preview:\n")
print(cleaned_pages[0]["text"][:2000])

Cleaned page 1 preview:

Metformin is one of the most widely prescribed oral antihyperglycemic agents. Its primary mechanism of action involves the activation of AMP-activated protein kinase (AMPK), a central metabolic regulator that promotes glucose uptake and fatty acid oxidation while inhibiting hepatic gluconeogenesis. Beyond its glycemic control, Metformin has been shown to improve cardiovascular outcomes and display anti-inflammatory properties. Recent studies also suggest potential anticancer effects through inhibition of the mTOR signaling pathway and suppression of tumor angiogenesis.

Clinical trials have demonstrated that combining Atorvastatin with Ezetimibe results in significant reductions in low-density lipoprotein cholesterol (LDL-C) levels compared to monotherapy. Ezetimibe acts by inhibiting the Niemann–Pick C1-like 1 (NPC1L1) transporter in the intestinal wall, reducing cholesterol absorption, while Atorvastatin inhibits hepatic HMG-CoA reductase, suppressing endogen

## 7. Split cleaned text into paragraph records

<span style="color:green">Store cleaned text as paragraph-level records so we can inspect training data more easily before tokenization.</span>

In [13]:
# ============================================================
# 7. Paragraph records
# ============================================================

def split_into_paragraph_records(cleaned_pages, min_chars=80):
    records = []
    seen = set()

    for page in cleaned_pages:
        paragraphs = re.split(r"\n\s*\n", page["text"])

        for paragraph_index, paragraph in enumerate(paragraphs, start=1):
            paragraph = paragraph.strip()

            if len(paragraph) < min_chars:
                continue

            # Remove exact duplicate paragraphs
            normalized_key = re.sub(r"\s+", " ", paragraph.lower()).strip()
            if normalized_key in seen:
                continue
            seen.add(normalized_key)

            records.append({
                "text": paragraph,
                "source_page": page["page"],
                "paragraph_id": paragraph_index,
                "char_count": len(paragraph),
            })

    return records

## 8. Save processed text for reproducibility

<span style="color:green">Save intermediate processed data. It helps with debugging, reproducibility, and auditability.</span>

In [14]:
# ============================================================
# 8. Save processed text
# ============================================================

paragraph_records = split_into_paragraph_records(
    cleaned_pages,
    min_chars=CONFIG["min_chars_per_paragraph"]
)

print(f"Total paragraph records: {len(paragraph_records)}")
raw_pages_path = os.path.join(CONFIG["processed_data_dir"], "pdf_pages_raw.jsonl")
paragraphs_path = os.path.join(CONFIG["processed_data_dir"], "pharma_paragraphs_cleaned.jsonl")

with open(raw_pages_path, "w", encoding="utf-8") as f:
    for item in pdf_pages:
        f.write(json.dumps(item, ensure_ascii=False) + "\n")

with open(paragraphs_path, "w", encoding="utf-8") as f:
    for item in paragraph_records:
        f.write(json.dumps(item, ensure_ascii=False) + "\n")

print(f"Saved raw pages to: {raw_pages_path}")
print(f"Saved cleaned paragraphs to: {paragraphs_path}")

Total paragraph records: 9
Saved raw pages to: /content/pharma_processed_data/pdf_pages_raw.jsonl
Saved cleaned paragraphs to: /content/pharma_processed_data/pharma_paragraphs_cleaned.jsonl


## 9. Create Hugging Face dataset

<span style="color:green">Transformers training works smoothly with Hugging Face Datasets, so we convert paragraph records into a Dataset object.</span>

In [15]:
# ============================================================
# 9. Create Hugging Face dataset
# ============================================================

if len(paragraph_records) < 2:
    raise ValueError("The extracted corpus is too small. Use a larger PDF or reduce min_chars_per_paragraph.")

text_dataset = Dataset.from_list(paragraph_records)
print(text_dataset)
print("\nExample row:\n")
print(text_dataset[0])

Dataset({
    features: ['text', 'source_page', 'paragraph_id', 'char_count'],
    num_rows: 9
})

Example row:

{'text': 'Metformin is one of the most widely prescribed oral antihyperglycemic agents. Its primary mechanism of action involves the activation of AMP-activated protein kinase (AMPK), a central metabolic regulator that promotes glucose uptake and fatty acid oxidation while inhibiting hepatic gluconeogenesis. Beyond its glycemic control, Metformin has been shown to improve cardiovascular outcomes and display anti-inflammatory properties. Recent studies also suggest potential anticancer effects through inhibition of the mTOR signaling pathway and suppression of tumor angiogenesis.', 'source_page': 1, 'paragraph_id': 1, 'char_count': 575}


## 10. Train / validation split

<span style="color:green">Even in small demo notebooks, we keep a validation split. But if the corpus is too small, the validation set may become empty after token packing.</span>

In [16]:
# ============================================================
# 10. Train / validation split
# ============================================================

split_dataset = text_dataset.train_test_split(
    test_size=CONFIG["test_size"],
    seed=CONFIG["seed"]
)

raw_datasets = DatasetDict({
    "train": split_dataset["train"],
    "validation": split_dataset["test"],
})

print(raw_datasets)

DatasetDict({
    train: Dataset({
        features: ['text', 'source_page', 'paragraph_id', 'char_count'],
        num_rows: 7
    })
    validation: Dataset({
        features: ['text', 'source_page', 'paragraph_id', 'char_count'],
        num_rows: 2
    })
})


## 11. Load tokenizer

<span style="color:green">The tokenizer converts text into token IDs.</span>

In [17]:
# ============================================================
# 11. Load tokenizer
# ============================================================

tokenizer = AutoTokenizer.from_pretrained(CONFIG["model_name"], use_fast=True)

# Many Llama-style models do not define a pad token
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

tokenizer.padding_side = "right"

print(f"Tokenizer loaded: {CONFIG['model_name']}")
print(f"Vocab size: {len(tokenizer)}")
print(f"Pad token: {tokenizer.pad_token} | id: {tokenizer.pad_token_id}")
print(f"EOS token: {tokenizer.eos_token} | id: {tokenizer.eos_token_id}")

config.json:   0%|          | 0.00/560 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/776 [00:00<?, ?B/s]

tokenizer.model:   0%|          | 0.00/500k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/1.84M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/414 [00:00<?, ?B/s]

Tokenizer loaded: TinyLlama/TinyLlama-1.1B-intermediate-step-1431k-3T
Vocab size: 32000
Pad token: </s> | id: 2
EOS token: </s> | id: 2


## 12. Tokenize and pack text into fixed-length blocks

<span style="color:green">The model trains on token sequences of fixed length like 512 tokens. We concatenate tokenized text and split it into equal-size blocks.</span>

<span style="color:green"><b>Important:</b> This is causal LM training, so <code>labels = input_ids</code>. Even though they are the same sequence, the model internally shifts them during loss calculation and learns to predict the <b>next token</b>.</span>

<span style="color:green"><b>Example:</b> If the sequence is <code>[Metformin, is, effective]</code>, the model learns <code>Metformin → is</code> and <code>Metformin is → effective</code>.</span>

In [18]:
# ============================================================
# 12. Tokenization and packing
# ============================================================

def tokenize_function(examples):
    return tokenizer(examples["text"])


def group_texts(examples):
    # Concatenate all tokenized examples
    concatenated_examples = {k: sum(examples[k], []) for k in examples.keys()}
    total_length = len(concatenated_examples["input_ids"])

    # Keep only full blocks
    total_length = (total_length // CONFIG["block_size"]) * CONFIG["block_size"]

    if total_length == 0:
        return {k: [] for k in concatenated_examples.keys()}

    result = {
        k: [t[i:i + CONFIG["block_size"]] for i in range(0, total_length, CONFIG["block_size"])]
        for k, t in concatenated_examples.items()
    }

    # For causal LM, labels are the same as input_ids
    result["labels"] = result["input_ids"].copy()
    return result

In [19]:
tokenized_datasets = raw_datasets.map(
    tokenize_function,
    batched=True,
    remove_columns=raw_datasets["train"].column_names,
    desc="Tokenizing text corpus",
)

final_datasets = tokenized_datasets.map(
    group_texts,
    batched=True,
    desc=f"Packing tokenized text into blocks of {CONFIG['block_size']}",
)

print(final_datasets)

Tokenizing text corpus:   0%|          | 0/7 [00:00<?, ? examples/s]

Tokenizing text corpus:   0%|          | 0/2 [00:00<?, ? examples/s]

Packing tokenized text into blocks of 512:   0%|          | 0/7 [00:00<?, ? examples/s]

Packing tokenized text into blocks of 512:   0%|          | 0/2 [00:00<?, ? examples/s]

DatasetDict({
    train: Dataset({
        features: ['input_ids', 'attention_mask', 'labels'],
        num_rows: 6
    })
    validation: Dataset({
        features: ['input_ids', 'attention_mask'],
        num_rows: 0
    })
})


In [20]:
if len(final_datasets["train"]) == 0:
    raise ValueError("No training blocks were created. Reduce block_size or use a larger corpus.")

In [21]:
sample = final_datasets["train"][0]
print("Keys:", sample.keys())
print("input_ids length:", len(sample["input_ids"]))
print("labels length:", len(sample["labels"]))
print("\nDecoded sample preview:\n")
print(tokenizer.decode(sample["input_ids"][:250]))

Keys: dict_keys(['input_ids', 'attention_mask', 'labels'])
input_ids length: 512
labels length: 512

Decoded sample preview:

<s> Pharma Domain Training Data - Page 5 Page 5 - AI in Drug Discovery and Pharmaceutical R&D; Pharma-domain corpus extension for custom fine-tuning and RAG experimentation. Educational content only; not medical advice. Target identification Artificial intelligence is increasingly used in pharmaceutical research to analyze genomics, transcriptomics, proteomics, disease phenotypes, chemical libraries, and clinical datasets. In target identification, machine learning models can prioritize genes or proteins that may play causal roles in disease biology. These predictions are strengthened when integrated with experimental validation, pathway analysis, human genetics, and disease-relevant biomarkers. Molecular screening In early discovery, deep learning can support virtual screening by predicting protein-ligand binding affinity, molecular properties, toxicity signals

## 13. Load base model

<span style="color:green">The <b>base model</b> is the original pretrained TinyLlama model. It already knows general language patterns from large-scale pretraining.</span>

<span style="color:green"> We do <b>not</b> fully retrain the base model. We freeze it and train a small LoRA adapter on top of it.</span>

<span style="color:green"> When using QLoRA, the base model is loaded in 4-bit format to reduce GPU memory usage.</span>

In [22]:
# ============================================================
# 13. Load base model
# ============================================================

import torch

use_cuda = torch.cuda.is_available()
print("CUDA available:", use_cuda)
if use_cuda:
    print("GPU:", torch.cuda.get_device_name(0))

CUDA available: True
GPU: Tesla T4


In [23]:
# Free memory before model load
import gc
gc.collect()
if use_cuda:
    torch.cuda.empty_cache()

In [24]:
from transformers import BitsAndBytesConfig, AutoModelForCausalLM
from peft import prepare_model_for_kbit_training

if use_cuda:
    bnb_config = BitsAndBytesConfig(
        load_in_4bit=True,  # Store model weights in 4-bit instead of full precision to fit large models on smaller GPUs
        bnb_4bit_quant_type="nf4",  # NF4 = NormalFloat4, a good 4-bit format for neural network weights
        bnb_4bit_compute_dtype=torch.float16,  # During forward/backward pass, compute using float16
        bnb_4bit_use_double_quant=True,  # Quantize quantization constants too, which further reduces memory
    )

    base_model = AutoModelForCausalLM.from_pretrained(
        CONFIG["model_name"],
        quantization_config=bnb_config,
        device_map="auto",
        trust_remote_code=True,
    )

    # Makes the quantized model safer and more stable for LoRA training
    base_model = prepare_model_for_kbit_training(base_model)

else:
    base_model = AutoModelForCausalLM.from_pretrained(
        CONFIG["model_name"],
        torch_dtype=torch.float32,  # Use normal full-precision loading on CPU
        trust_remote_code=True,
    )

# Turn off cache during training because it can conflict with training/memory settings
base_model.config.use_cache = False

print("Base model loaded successfully.")

model.safetensors:   0%|          | 0.00/4.40G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

/usr/local/lib/python3.12/dist-packages/bitsandbytes/backends/cuda/ops.py:213: FutureWarning: _check_is_size will be removed in a future PyTorch release along with guard_size_oblivious.     Use _check(i >= 0) instead.
  torch._check_is_size(blocksize)


generation_config.json:   0%|          | 0.00/129 [00:00<?, ?B/s]

Base model loaded successfully.


## 14. Add LoRA adapter

<span style="color:green"> A <b>LoRA adapter</b> is a small trainable module inserted into selected layers of the base model.</span>

<span style="color:green">Instead of updating all 1.1B model weights, we update only a small number of adapter weights. This makes training cheaper and faster.</span>

In [25]:
# ============================================================
# 14. Apply LoRA
# ============================================================

# Create the LoRA configuration.
# LoRA adds small trainable adapter layers into selected parts of the base model.
# Instead of updating all model weights, we only train these lightweight adapter weights.
lora_config = LoraConfig(
    task_type=TaskType.CAUSAL_LM,  # This LoRA adapter is for causal language modeling
    r=CONFIG["lora_r"],  # LoRA rank; controls adapter capacity
    lora_alpha=CONFIG["lora_alpha"],  # Scaling factor; controls how strongly LoRA affects the model
    lora_dropout=CONFIG["lora_dropout"],  # Dropout inside LoRA layers to reduce overfitting
    bias="none",  # Do not train bias terms; keeps adapter smaller and simpler
    target_modules=[
        "q_proj",  # Query projection in attention
        "k_proj",  # Key projection in attention
        "v_proj",  # Value projection in attention
        "o_proj",  # Output projection in attention
        "gate_proj",  # Gating projection in feed-forward network
        "up_proj",  # Up projection in feed-forward network
        "down_proj",  # Down projection in feed-forward network
    ],
)

# Attach the LoRA adapter to the base model.
# After this step, the model becomes a PEFT model (base model + LoRA adapter).
model = get_peft_model(base_model, lora_config)

# Print how many parameters are trainable.
# This helps confirm that only a small percentage of weights will be updated.
model.print_trainable_parameters()

trainable params: 12,615,680 || all params: 1,112,664,064 || trainable%: 1.1338


## 15. Data collator

<span style="color:green"> The data collator converts tokenized examples into mini-batches during training.</span>

<span style="color:green"><code>mlm=False</code> means we are doing <b>causal language modeling</b>, not masked language modeling like BERT.</span>

In [26]:
# ============================================================
# 15. Data collator
# ============================================================

data_collator = DataCollatorForLanguageModeling(
    tokenizer=tokenizer,
    mlm=False
)

## 16. Training arguments

<span style="color:green"> These settings control how training runs: batch size, learning rate, saving frequency, logging frequency, and mixed precision.</span>

In [27]:
# ============================================================
# 16. Training arguments
# ============================================================

# If validation data exists, run evaluation during training.
# Otherwise, disable evaluation completely.
eval_mode = "steps" if len(final_datasets["validation"]) > 0 else "no"

training_args = TrainingArguments(
    output_dir=CONFIG["output_dir"],  # Folder where checkpoints, logs, and outputs will be saved

    num_train_epochs=CONFIG["num_train_epochs"],  # Number of full passes through the training dataset

    max_steps=CONFIG["max_steps"],  # If > 0, training stops after this many steps; -1 means use epochs

    per_device_train_batch_size=CONFIG["per_device_train_batch_size"],  # Training batch size per device

    per_device_eval_batch_size=CONFIG["per_device_eval_batch_size"],  # Evaluation batch size per device

    gradient_accumulation_steps=CONFIG["gradient_accumulation_steps"],  # Accumulate gradients before optimizer update

    learning_rate=CONFIG["learning_rate"],  # Step size used by the optimizer to update weights

    warmup_steps=5,  # Gradually increase learning rate at the beginning for stability

    weight_decay=CONFIG["weight_decay"],  # Regularization to reduce overfitting

    logging_steps=CONFIG["logging_steps"],  # Print/log training progress every N steps

    eval_steps=CONFIG["eval_steps"],  # Run validation every N steps when evaluation is enabled

    save_steps=CONFIG["save_steps"],  # Save a checkpoint every N steps

    save_total_limit=CONFIG["save_total_limit"],  # Keep only the most recent checkpoints

    fp16=use_cuda,  # Use float16 mixed precision on GPU to save memory and speed up training

    bf16=False,  # Do not use bfloat16 here

    report_to="none",  # Disable logging to external services like WandB

    remove_unused_columns=False,  # Keep all dataset columns because causal LM uses custom fields

    eval_strategy=eval_mode,  # Evaluation mode: "steps" if validation exists, otherwise "no"
)
print(training_args)

TrainingArguments(
accelerator_config={'split_batches': False, 'dispatch_batches': None, 'even_batches': True, 'use_seedable_sampler': True, 'non_blocking': False, 'gradient_accumulation_kwargs': None, 'use_configured_state': False},
adam_beta1=0.9,
adam_beta2=0.999,
adam_epsilon=1e-08,
auto_find_batch_size=False,
average_tokens_across_devices=True,
batch_eval_metrics=False,
bf16=False,
bf16_full_eval=False,
data_seed=None,
dataloader_drop_last=False,
dataloader_num_workers=0,
dataloader_persistent_workers=False,
dataloader_pin_memory=True,
dataloader_prefetch_factor=None,
ddp_backend=None,
ddp_broadcast_buffers=None,
ddp_bucket_cap_mb=None,
ddp_find_unused_parameters=None,
ddp_static_graph=None,
ddp_timeout=1800,
debug=[],
deepspeed=None,
disable_tqdm=False,
do_eval=False,
do_predict=False,
do_train=False,
enable_jit_checkpoint=False,
eval_accumulation_steps=None,
eval_delay=0,
eval_do_concat_batches=True,
eval_on_start=False,
eval_steps=10,
eval_strategy=IntervalStrategy.NO,
eval_use

## 17. Build Trainer

<span style="color:green"> The Trainer handles the training loop: batching, forward pass, loss computation, backpropagation, optimizer step, checkpoint saving, and logging.</span>

In [28]:
# ============================================================
# 17. Build Trainer
# ============================================================

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=final_datasets["train"],
    eval_dataset=final_datasets["validation"] if len(final_datasets["validation"]) > 0 else None,
    data_collator=data_collator,
    processing_class=tokenizer if "processing_class" in Trainer.__init__.__code__.co_varnames else None,
)

## 18. Start training

<span style="color:green">During training, the model sees pharma token sequences and learns to predict the next token.</span>

<span style="color:green"><b>Important:</b> Only the <b>LoRA adapter</b> is updated. The base model remains frozen.</span>

In [29]:
# ============================================================
# 18. Train
# ============================================================

train_result = trainer.train()

print("Training completed.")
print(train_result)

[transformers] The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'pad_token_id': 2}.
/usr/local/lib/python3.12/dist-packages/torch/_dynamo/eval_frame.py:1263: UserWarning: torch.utils.checkpoint: the use_reentrant parameter should be passed explicitly. Starting in PyTorch 2.9, calling checkpoint without use_reentrant will raise an exception. use_reentrant=False is recommended, but if you need to preserve the current default behavior, you can pass use_reentrant=True. Refer to docs for more details on the differences between the two variants.
  return fn(*args, **kwargs)


Step,Training Loss


Training completed.
TrainOutput(global_step=3, training_loss=2.1406095822652182, metrics={'train_runtime': 19.8075, 'train_samples_per_second': 0.909, 'train_steps_per_second': 0.151, 'total_flos': 57901993426944.0, 'train_loss': 2.1406095822652182, 'epoch': 3.0})


## 19. Save LoRA adapter

<span style="color:green"> We save only the adapter weights, not the full base model. This is why LoRA is storage-efficient.</span>

In [30]:
# ============================================================
# 19. Save adapter and tokenizer
# ============================================================

trainer.model.save_pretrained(CONFIG["adapter_dir"])
tokenizer.save_pretrained(CONFIG["adapter_dir"])

print(f"LoRA adapter saved to: {CONFIG['adapter_dir']}")
print("Saved files:")
print(os.listdir(CONFIG["adapter_dir"]))

LoRA adapter saved to: /content/pharma_tinyllama_lora_adapter
Saved files:
['tokenizer_config.json', 'adapter_config.json', 'README.md', 'adapter_model.safetensors', 'tokenizer.json']


## 20. Push LoRA adapter to Hugging Face Hub

In [40]:
# ============================================================
# 20. Push LoRA adapter to Hugging Face Hub
# ============================================================

repo_id = "ssuvetha/pharma-tinyllama-domain-lora"

model.push_to_hub(repo_id, private=True, token=WRITE_TOKEN)
tokenizer.push_to_hub(repo_id, private=True, token=WRITE_TOKEN)

print(f"LoRA adapter pushed to: https://huggingface.co/{repo_id}")

Processing Files (0 / 0)      : |          |  0.00B /  0.00B            

New Data Upload               : |          |  0.00B /  0.00B            

  ...adapter_model.safetensors: 100%|##########| 50.5MB / 50.5MB            

No files have been modified since last commit. Skipping to prevent empty commit.
No files have been modified since last commit. Skipping to prevent empty commit.


LoRA adapter pushed to: https://huggingface.co/ssuvetha/pharma-tinyllama-domain-lora


## 21. Reload base model + adapter for inference

<span style="color:green"> For inference, we load:</span>

<span style="color:green">1. The original base model</span>  
<span style="color:green">2. The saved LoRA adapter</span>  
<span style="color:green">3. Combine them into one inference model</span>

<span style="color:green">This is how PEFT works in practice: base model + adapter together produce domain-adapted output.</span>

In [41]:
# ============================================================
# 21. Reload for inference
# ============================================================

# Clean up memory
del trainer
try:
    del model
    del base_model
except NameError:
    pass

gc.collect()
if use_cuda:
    torch.cuda.empty_cache()

In [42]:
if use_cuda:
    reload_bnb_config = BitsAndBytesConfig(
        load_in_4bit=True,
        bnb_4bit_quant_type="nf4",
        bnb_4bit_compute_dtype=torch.float16,
        bnb_4bit_use_double_quant=True,
    )

    inference_base_model = AutoModelForCausalLM.from_pretrained(
        CONFIG["model_name"],
        quantization_config=reload_bnb_config,
        device_map="auto",
        trust_remote_code=True,
    )
else:
    inference_base_model = AutoModelForCausalLM.from_pretrained(
        CONFIG["model_name"],
        torch_dtype=torch.float32,
        trust_remote_code=True,
    )

inference_tokenizer = AutoTokenizer.from_pretrained(CONFIG["adapter_dir"], use_fast=True)

if inference_tokenizer.pad_token is None:
    inference_tokenizer.pad_token = inference_tokenizer.eos_token

inference_model = PeftModel.from_pretrained(
    inference_base_model,
    CONFIG["adapter_dir"]
)

inference_model.eval()

print("Base model + LoRA adapter loaded successfully for inference.")

Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

/usr/local/lib/python3.12/dist-packages/bitsandbytes/backends/cuda/ops.py:213: FutureWarning: _check_is_size will be removed in a future PyTorch release along with guard_size_oblivious.     Use _check(i >= 0) instead.
  torch._check_is_size(blocksize)


Base model + LoRA adapter loaded successfully for inference.


## 22. Inference helper

<span style="color:green">This notebook is doing <b>non-instruction fine-tuning</b>, so inference should use <b>continuation-style prompts</b>, not chat-style Q&A prompts.</span>

In [43]:
import torch

# ============================================================
# 21. Inference helper
# ============================================================

def generate_completion(prompt, max_new_tokens=120):
    device = "cuda" if torch.cuda.is_available() else "cpu"

    inputs = inference_tokenizer(prompt, return_tensors="pt")
    inputs = {k: v.to(device) for k, v in inputs.items()}

    with torch.no_grad():
        outputs = inference_model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            do_sample=True,
            temperature=0.7,
            top_p=0.9,
            repetition_penalty=1.1,
            pad_token_id=inference_tokenizer.eos_token_id,
            eos_token_id=inference_tokenizer.eos_token_id,
        )

    return inference_tokenizer.decode(outputs[0], skip_special_tokens=True)

## 23. Test text continuation

<span style="color:green">We provide a partial pharma sentence and ask the model to continue it. This matches causal LM behavior.</span>

In [44]:
# ============================================================
# 22. Test generation
# ============================================================

prompts = [
    "Metformin is one of the most widely prescribed oral antihyperglycemic agents",
    "Clinical trials have demonstrated that combining Atorvastatin with Ezetimibe",
    "Artificial intelligence is transforming pharmaceutical research by accelerating",
]

for prompt in prompts:
    print("=" * 100)
    print("PROMPT:")
    print(prompt)
    print("\nMODEL CONTINUATION:")
    print(generate_completion(prompt, max_new_tokens=120))
    print()

PROMPT:
Metformin is one of the most widely prescribed oral antihyperglycemic agents

MODEL CONTINUATION:


[transformers] Both `max_new_tokens` (=120) and `max_length`(=2048) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
/usr/local/lib/python3.12/dist-packages/bitsandbytes/backends/cuda/ops.py:468: FutureWarning: _check_is_size will be removed in a future PyTorch release along with guard_size_oblivious.     Use _check(i >= 0) instead.
  torch._check_is_size(blocksize)
[transformers] Both `max_new_tokens` (=120) and `max_length`(=2048) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


Metformin is one of the most widely prescribed oral antihyperglycemic agents in the world. It is a sulfonylurea class drug, which has been used to treat type 2 diabetes for over 30 years. However, despite its proven efficacy and safety, the mechanism by which it works remains unknown. A recent study from the National Institutes of Health (NIH) reveals that Metformin can directly affect the cellular processes that regulate the production of insulin. This finding could lead to the development of new drugs targeting these processes.
"These findings are very important because they suggest that Met

PROMPT:
Clinical trials have demonstrated that combining Atorvastatin with Ezetimibe

MODEL CONTINUATION:


[transformers] Both `max_new_tokens` (=120) and `max_length`(=2048) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


Clinical trials have demonstrated that combining Atorvastatin with Ezetimibe, a statin drug that reduces cholesterol and LDL cholesterol, lowers LDL-C by 39% compared to statins alone.
The study was published online in the New England Journal of Medicine.
Atorvastatin is a lipid (fat) lowering drug used for primary prevention of cardiovascular disease, as well as secondary prevention of coronary artery disease, strokes and other types of heart attacks.
Atrial fibrillation is an irregular heartbeat that causes unpredict

PROMPT:
Artificial intelligence is transforming pharmaceutical research by accelerating

MODEL CONTINUATION:
Artificial intelligence is transforming pharmaceutical research by accelerating drug discovery, reducing time-to-market and increasing the efficiency of R&D operations.
Industrial IoT is driving the next wave of innovation in healthcare as a growing number of devices and apps are connected to create smart environments for better patient care.
MedTech IoT is revol

## 23. Optional merge step

<span style="color:green"><b>Concept:</b> Merging means combining the LoRA adapter weights into the base model weights.</span>

<span style="color:green"><b>Use merge when:</b> you want a single standalone model for deployment.</span>

<span style="color:green"><b>Skip merge when:</b> you want to keep the adapter separate and lightweight.</span>

In [ ]:
# ============================================================
# 23. Optional merge step
# ============================================================
# Uncomment this block only if you want a merged standalone model.

# merged_model_dir = "/content/pharma_tinyllama_merged_model"
# merged_model = inference_model.merge_and_unload()
# merged_model.save_pretrained(merged_model_dir)
# inference_tokenizer.save_pretrained(merged_model_dir)
# print(f"Merged model saved to: {merged_model_dir}")

## 24. Final summary

<span style="color:green"><b>What this notebook did:</b></span>

- Extracted pharma text from a PDF
- Cleaned and normalized it
- Built a Hugging Face dataset
- Tokenized and packed text into 512-token blocks
- Loaded TinyLlama as the base model
- Added a LoRA adapter
- Trained only the adapter with QLoRA-friendly settings
- Saved the adapter
- Reloaded the base model + adapter
- Generated domain-style continuations



## ============================================================
## 25. Mount Google Drive
## ============================================================

<span style="color:green"><b>Why this step?</b> Files stored only in the Colab runtime (for example under <code>/content/...</code>) may be lost when the session ends or the runtime restarts.</span>

<span style="color:green"><b>Goal:</b> Mount Google Drive so we can copy important outputs to persistent storage before closing the notebook.</span>

In [45]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive



## 25. Backup important folders to Google Drive
#

<span style="color:green"><b>Why this step?</b> The trained LoRA adapter, processed dataset, and training outputs are important files that should be saved outside the temporary Colab environment.</span>

<span style="color:green"><b>This step copies:</b></span>

<span style="color:green">
- <code>processed_data_dir</code><br>
- <code>adapter_dir</code><br>
- <code>output_dir</code>
</span>

<span style="color:green"><b>Result:</b> A backup copy will be created in Google Drive so the files can be reused in future sessions.</span>

In [46]:
import os
import shutil

drive_save_dir = "/content/drive/MyDrive/pharma_finetune_backup"
os.makedirs(drive_save_dir, exist_ok=True)

folders_to_backup = [
    CONFIG["processed_data_dir"],
    CONFIG["adapter_dir"],
    CONFIG["output_dir"],
]

for folder in folders_to_backup:
    if os.path.exists(folder):
        dest = os.path.join(drive_save_dir, os.path.basename(folder))
        if os.path.exists(dest):
            shutil.rmtree(dest)
        shutil.copytree(folder, dest)
        print(f"Copied: {folder} -> {dest}")
    else:
        print(f"Skipped (not found): {folder}")

Copied: /content/pharma_processed_data -> /content/drive/MyDrive/pharma_finetune_backup/pharma_processed_data
Copied: /content/pharma_tinyllama_lora_adapter -> /content/drive/MyDrive/pharma_finetune_backup/pharma_tinyllama_lora_adapter
Copied: /content/pharma_tinyllama_lora_output -> /content/drive/MyDrive/pharma_finetune_backup/pharma_tinyllama_lora_output



## 26. Verify backup


<span style="color:green"><b>Why this step?</b> After copying files to Google Drive, we should confirm that the backup folders are present.</span>

<span style="color:green"><b>Goal:</b> Check that the expected folders appear in the Drive backup location before closing the session.</span>

In [47]:
print(os.listdir("/content/drive/MyDrive/pharma_finetune_backup"))

['pharma_processed_data', 'pharma_tinyllama_lora_adapter', 'pharma_tinyllama_lora_output']
